# 04 — Document Processing (Phase 4)

**Project:** DGCA Agentic AI Regulatory Intelligence & Monitoring System
**Phase:** 4 of 8 — PDF Download, Validation, Text Extraction & OCR

## 1. Project Overview

Phase 3 tells us *which* publications need processing (`get_pending_notifications()`
and friends). This notebook does the actual document work for a publication's
`pdf_url`: download it, make sure it's really a usable PDF, extract its text,
detect whether it's a scanned image rather than real text, and OCR it if
needed and possible.

```
Publication record (pdf_url)
        |
        v
download_pdf()          -- retry/backoff, raises DownloadError on failure
        |
        v
validate_pdf_bytes()    -- magic bytes + can PyMuPDF actually open it
        |
        v
save to data/documents/{publication_id}.pdf
        |
        v
doc.is_encrypted?  ------ yes --> status = password_protected (stop)
        | no
doc.page_count == 0? ---- yes --> status = empty (stop)
        | no
extract_text()  -- PyMuPDF primary, pdfplumber fallback
        |
        v
is_likely_scanned()  -- chars-per-page below threshold despite pages existing
        |
        +-- no  -> status = success
        +-- yes, Tesseract available   -> OCR -> status = ocr_success
        +-- yes, Tesseract unavailable -> status = needs_ocr (NOT a crash, NOT a fake success)
        v
save extracted text -> data/documents/{publication_id}.txt
        v
document_hash (SHA-256 of the raw PDF bytes) written back into Phase 3's
PostgreSQL `publications` table -- closing the TODO Phase 3 left open.
```

**Scope of this phase:** document-level processing only. Comparing an old
version of a document against a new one is Phase 5; summarizing/analyzing
content with an LLM is Phase 6.


## 2. OCR Availability Check

OCR needs the separate **Tesseract** binary, not just the `pytesseract`
Python wrapper -- and it is confirmed **not installed** in this environment
(checked before writing any code: `where tesseract` found nothing). Per this
project's own security/side-effect rules, an external executable like that
is not something to silently download and install -- that decision belongs
to whoever runs this notebook.

So OCR support here is **availability-gated**: `pytesseract` is imported and
used when the Tesseract binary is actually reachable; when it isn't, scanned
documents are classified `needs_ocr` -- logged clearly, saved for later,
never silently dropped or faked as successfully processed. This cell's
result (`OCR_AVAILABLE`) controls that branch for the rest of the notebook.

**To enable OCR later:** install Tesseract yourself (e.g. the official
[UB-Mannheim Windows build](https://github.com/UB-Mannheim/tesseract/wiki),
or `winget install UB-Mannheim.TesseractOCR`), then re-run this notebook --
no code changes needed.


In [ ]:
try:
    import pytesseract
    from PIL import Image
    _PYTESSERACT_SDK_AVAILABLE = True
except ImportError:
    _PYTESSERACT_SDK_AVAILABLE = False


def check_ocr_availability() -> bool:
    if not _PYTESSERACT_SDK_AVAILABLE:
        print("pytesseract package not installed -- OCR unavailable.")
        return False
    try:
        version = pytesseract.get_tesseract_version()
        print(f"Tesseract OCR binary found: version {version}")
        return True
    except Exception as exc:
        print(f"pytesseract is installed, but the Tesseract OCR binary is NOT available ({exc.__class__.__name__}).")
        print("OCR will be gated OFF for this run -- scanned documents will be classified \'needs_ocr\' instead of processed.")
        return False


OCR_AVAILABLE = check_ocr_availability()


## 3. Imports

In [ ]:
import hashlib
import io
import json
import logging
from psycopg import Connection
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Callable, Dict, List, Optional, TypeVar

import requests
import pymupdf
import pdfplumber

print("Imports OK")


## 4. Configuration

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
LOG_DIR = PROJECT_ROOT / "logs"

DOCUMENTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

REQUEST_TIMEOUT_SECONDS = 30
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120 Safari/537.36"
)
MIN_VALID_PDF_SIZE_BYTES = 100
SCANNED_CHARS_PER_PAGE_THRESHOLD = 20  # avg extracted chars/page below this => treat as scanned

print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"DOCUMENTS_DIR  : {DOCUMENTS_DIR}")
print(f"DB_PATH        : PostgreSQL (DATABASE_URL)")

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / "config" / ".env")


## 5. Logging

In [ ]:
LOG_FILE = LOG_DIR / f"document_processing_{datetime.now(timezone.utc).strftime('%Y%m%d')}.log"

logger = logging.getLogger("dgca_document_processing")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_formatter = logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")

_console_handler = logging.StreamHandler()
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)

_file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

logger.propagate = False

logger.info("Document processing logger initialized.")
logger.info(f"Log file: {LOG_FILE}")
logger.info(f"OCR available: {OCR_AVAILABLE}")


## 6. Data Model

`DocumentStatus` gives every failure mode from the project brief's list its
own explicit, distinguishable outcome, rather than lumping everything into a
generic "failed".


In [ ]:
class DocumentStatus(str, Enum):
    SUCCESS = "success"
    OCR_SUCCESS = "ocr_success"
    NEEDS_OCR = "needs_ocr"
    EMPTY = "empty"
    PASSWORD_PROTECTED = "password_protected"
    CORRUPTED = "corrupted"
    DOWNLOAD_FAILED = "download_failed"
    FAILED = "failed"


@dataclass
class DocumentProcessingResult:
    publication_id: str
    pdf_url: Optional[str]
    status: str
    local_path: Optional[str] = None
    file_size_bytes: Optional[int] = None
    page_count: Optional[int] = None
    extraction_method: Optional[str] = None
    is_scanned: bool = False
    text_char_count: int = 0
    text_path: Optional[str] = None
    document_hash: Optional[str] = None
    error: Optional[str] = None
    processed_at: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


print("Data model ready.")


## 7. Download & Validation

Two-layer validation: a cheap magic-bytes/size check first, then an actual
parse attempt via PyMuPDF -- which is what catches a file that merely
*starts* with `%PDF-` but is truncated or structurally corrupted.


In [ ]:
T = TypeVar("T")


def retry_with_backoff(fn: Callable[[], T], max_attempts: int = 3, base_delay: float = 1.0) -> T:
    last_exc: Optional[Exception] = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except Exception as exc:
            last_exc = exc
            if attempt == max_attempts:
                break
            delay = base_delay * (2 ** (attempt - 1))
            logger.warning(f"Attempt {attempt}/{max_attempts} failed ({exc.__class__.__name__}). Retrying in {delay:.1f}s...")
            time.sleep(delay)
    raise last_exc


class DownloadError(Exception):
    pass


class ValidationError(Exception):
    pass


def download_pdf(pdf_url: str) -> bytes:
    headers = {"User-Agent": USER_AGENT}

    def _do_get():
        resp = requests.get(pdf_url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
        resp.raise_for_status()
        return resp

    try:
        response = retry_with_backoff(_do_get, max_attempts=3)
    except requests.RequestException as exc:
        raise DownloadError(f"{exc.__class__.__name__}: {exc}") from exc
    return response.content


def validate_pdf_bytes(raw_bytes: bytes) -> None:
    if len(raw_bytes) < MIN_VALID_PDF_SIZE_BYTES:
        raise ValidationError(f"File too small to be a valid PDF ({len(raw_bytes)} bytes).")
    if not raw_bytes.startswith(b"%PDF-"):
        raise ValidationError("File does not start with the %PDF- magic bytes.")
    try:
        doc = pymupdf.open(stream=raw_bytes, filetype="pdf")
        doc.close()
    except Exception as exc:
        raise ValidationError(f"PyMuPDF could not open the file ({exc.__class__.__name__}): {exc}") from exc


print("Download + validation functions ready.")


## 8. Text Extraction

PyMuPDF first (fast, handles the vast majority of real-world PDFs). If it
comes back empty, pdfplumber is tried as a fallback -- some PDFs are laid
out in ways one library handles better than the other. If both come back
empty, that's a strong signal the document is scanned rather than broken.


In [ ]:
def extract_text_pymupdf(doc: "pymupdf.Document") -> str:
    return "\n".join(page.get_text() for page in doc)


def extract_text_pdfplumber(path: Path) -> str:
    text_parts = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text_parts.append(page.extract_text() or "")
    return "\n".join(text_parts)


def is_likely_scanned(text: str, page_count: int) -> bool:
    if page_count == 0:
        return False
    avg_chars_per_page = len(text.strip()) / page_count
    return avg_chars_per_page < SCANNED_CHARS_PER_PAGE_THRESHOLD


print("Text extraction functions ready.")


## 9. OCR (used only when `OCR_AVAILABLE` is True)

Renders each page to an image via PyMuPDF and runs it through Tesseract.
Never called unless Section 2 confirmed the binary is actually reachable.


In [ ]:
def ocr_extract(doc: "pymupdf.Document", dpi: int = 200) -> str:
    text_parts = []
    zoom = dpi / 72
    matrix = pymupdf.Matrix(zoom, zoom)
    for page in doc:
        pix = page.get_pixmap(matrix=matrix)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        text_parts.append(pytesseract.image_to_string(img))
    return "\n".join(text_parts)


print("OCR function ready" + (" (will be used when needed)." if OCR_AVAILABLE else " (defined, but gated off -- Tesseract unavailable)."))


## 10. Orchestrator — `process_document()`

Ties every stage together. `raw_bytes_override` lets the test cells below
feed synthetic PDF bytes directly (bypassing the network call) while
exercising exactly the same downstream validation/extraction/OCR-gating
logic a real download would go through. Every failure mode returns a
structured `DocumentProcessingResult` — this function never raises.


In [ ]:
def compute_document_hash(raw_bytes: bytes) -> str:
    return hashlib.sha256(raw_bytes).hexdigest()


def process_document(
    publication_id: str,
    pdf_url: Optional[str],
    raw_bytes_override: Optional[bytes] = None,
) -> DocumentProcessingResult:
    logger.info(f"Processing document for {publication_id}")

    if not pdf_url and raw_bytes_override is None:
        logger.error(f"[{publication_id}] No pdf_url provided -- nothing to process.")
        return DocumentProcessingResult(
            publication_id, pdf_url, DocumentStatus.DOWNLOAD_FAILED.value, error="No pdf_url provided."
        )

    try:
        raw_bytes = raw_bytes_override if raw_bytes_override is not None else download_pdf(pdf_url)
    except DownloadError as exc:
        logger.error(f"[{publication_id}] Download failed: {exc}")
        return DocumentProcessingResult(publication_id, pdf_url, DocumentStatus.DOWNLOAD_FAILED.value, error=str(exc))

    try:
        validate_pdf_bytes(raw_bytes)
    except ValidationError as exc:
        logger.error(f"[{publication_id}] Validation failed: {exc}")
        return DocumentProcessingResult(
            publication_id, pdf_url, DocumentStatus.CORRUPTED.value,
            file_size_bytes=len(raw_bytes), error=str(exc),
        )

    local_path = DOCUMENTS_DIR / f"{publication_id}.pdf"
    local_path.write_bytes(raw_bytes)
    document_hash = compute_document_hash(raw_bytes)
    logger.info(f"[{publication_id}] Saved to {local_path} ({len(raw_bytes)} bytes).")

    doc = None
    try:
        doc = pymupdf.open(stream=raw_bytes, filetype="pdf")

        if doc.is_encrypted:
            logger.warning(f"[{publication_id}] PDF is password-protected -- cannot extract text.")
            return DocumentProcessingResult(
                publication_id, pdf_url, DocumentStatus.PASSWORD_PROTECTED.value,
                local_path=str(local_path), file_size_bytes=len(raw_bytes),
                page_count=doc.page_count, document_hash=document_hash,
            )

        if doc.page_count == 0:
            # Defensive: a spec-conformant PDF always has >=1 page (PyMuPDF
            # itself refuses to save a zero-page document), but a
            # maliciously/oddly crafted file could still claim this.
            logger.warning(f"[{publication_id}] PDF has zero pages.")
            return DocumentProcessingResult(
                publication_id, pdf_url, DocumentStatus.EMPTY.value,
                local_path=str(local_path), file_size_bytes=len(raw_bytes),
                page_count=0, document_hash=document_hash,
            )

        text = extract_text_pymupdf(doc)
        method = "pymupdf"
        if not text.strip():
            try:
                text = extract_text_pdfplumber(local_path)
                method = "pdfplumber"
            except Exception as exc:
                logger.warning(f"[{publication_id}] pdfplumber fallback also failed: {exc}")

        has_images = any(page.get_images() for page in doc)

        if not text.strip() and not has_images:
            logger.warning(f"[{publication_id}] No text and no images on any page -- blank/empty document.")
            return DocumentProcessingResult(
                publication_id, pdf_url, DocumentStatus.EMPTY.value,
                local_path=str(local_path), file_size_bytes=len(raw_bytes),
                page_count=doc.page_count, document_hash=document_hash,
            )

        scanned = is_likely_scanned(text, doc.page_count)

        if scanned:
            if OCR_AVAILABLE:
                logger.info(f"[{publication_id}] Detected as scanned -- running OCR.")
                ocr_text = ocr_extract(doc)
                text_path = DOCUMENTS_DIR / f"{publication_id}.txt"
                text_path.write_text(ocr_text, encoding="utf-8")
                logger.info(f"[{publication_id}] OCR complete: {len(ocr_text)} characters extracted.")
                return DocumentProcessingResult(
                    publication_id, pdf_url, DocumentStatus.OCR_SUCCESS.value,
                    local_path=str(local_path), file_size_bytes=len(raw_bytes), page_count=doc.page_count,
                    extraction_method="ocr", is_scanned=True, text_char_count=len(ocr_text),
                    text_path=str(text_path), document_hash=document_hash,
                )
            logger.warning(f"[{publication_id}] Detected as scanned, but OCR is unavailable in this environment.")
            return DocumentProcessingResult(
                publication_id, pdf_url, DocumentStatus.NEEDS_OCR.value,
                local_path=str(local_path), file_size_bytes=len(raw_bytes), page_count=doc.page_count,
                is_scanned=True, document_hash=document_hash,
                error="Scanned document; Tesseract OCR binary not available in this environment.",
            )

        text_path = DOCUMENTS_DIR / f"{publication_id}.txt"
        text_path.write_text(text, encoding="utf-8")
        logger.info(f"[{publication_id}] Text extracted via {method}: {len(text)} characters.")
        return DocumentProcessingResult(
            publication_id, pdf_url, DocumentStatus.SUCCESS.value,
            local_path=str(local_path), file_size_bytes=len(raw_bytes), page_count=doc.page_count,
            extraction_method=method, is_scanned=False, text_char_count=len(text),
            text_path=str(text_path), document_hash=document_hash,
        )
    except Exception as exc:
        logger.error(f"[{publication_id}] Unexpected error during processing: {exc.__class__.__name__}: {exc}")
        return DocumentProcessingResult(
            publication_id, pdf_url, DocumentStatus.FAILED.value,
            local_path=str(local_path), file_size_bytes=len(raw_bytes),
            document_hash=document_hash, error=str(exc),
        )
    finally:
        if doc is not None:
            doc.close()


print("process_document() ready.")


## 11. Phase 3 Integration — writing `document_hash` back

Phase 3 left `document_hash` as `NULL` with a note that Phase 4 would fill
it in once PDFs were actually downloaded and hashed. This is that connection
-- deliberately narrow: it only ever writes `document_hash`, never touches
Phase 3's notification-lifecycle fields (`status`, `email_sent`, ...).


In [ ]:
from pipeline.storage import get_connection


def update_document_hash(conn: Connection, publication_id: str, document_hash: str) -> None:
    conn.execute(
        "UPDATE publications SET document_hash = %s WHERE publication_id = %s",
        (document_hash, publication_id),
    )
    conn.commit()


print("Phase 3 integration helpers ready.")


## 12. Synthetic Test Fixtures

Real DGCA PDFs cover the "normal, text-based" and "scanned" cases (see
Sections 13 and 16). Empty, corrupted, and password-protected PDFs are hard
to find on demand from a real website, so these three are generated
synthetically with PyMuPDF -- standard practice for exercising specific edge
cases deterministically.

**Note on "empty":** PyMuPDF refuses to save a genuinely zero-page document
(`ValueError: cannot save with zero pages`) -- correctly, since a valid PDF
must have at least one page. So "empty" here means the realistic version of
that case: **a page that exists but has no text and no images** -- a blank
page. `process_document()` (Section 10) treats "no text and no images" as
`empty`, distinct from `needs_ocr`/`scanned` (which has images but no text
layer).


In [ ]:
def make_empty_pdf_bytes() -> bytes:
    doc = pymupdf.open()
    doc.new_page()  # one blank page: no text, no images
    buf = io.BytesIO()
    doc.save(buf)
    doc.close()
    return buf.getvalue()


def make_corrupted_pdf_bytes() -> bytes:
    # Starts with a valid PDF header (passes the cheap magic-byte check) but
    # has no valid structure after it -- PyMuPDF should fail to open it.
    return b"%PDF-1.4\n" + b"this is not a real PDF body, just garbage. " * 10


def make_password_protected_pdf_bytes(password: str = "secret123") -> bytes:
    doc = pymupdf.open()
    page = doc.new_page()
    page.insert_text((72, 72), "This content is protected.")
    buf = io.BytesIO()
    doc.save(buf, encryption=pymupdf.PDF_ENCRYPT_AES_256, owner_pw=password, user_pw=password)
    doc.close()
    return buf.getvalue()


_empty_bytes = make_empty_pdf_bytes()
_corrupted_bytes = make_corrupted_pdf_bytes()
_password_bytes = make_password_protected_pdf_bytes()

print(f"Synthetic empty PDF            : {len(_empty_bytes)} bytes")
print(f"Synthetic corrupted PDF        : {len(_corrupted_bytes)} bytes")
print(f"Synthetic password-protected PDF: {len(_password_bytes)} bytes")


## 13. Test 1 — Real DGCA PDF (normal, text-based)

A genuine DGCA regulation PDF (already confirmed text-based during Phase 2
testing), downloaded for real.


In [ ]:
conn = get_connection()
real_row = conn.execute(
    "SELECT publication_id, pdf_url FROM publications WHERE publication_id = %s",
    ("regulation-yv5UwjfEILZBCtSQvOHdA",),
).fetchone()

result_normal = process_document(real_row["publication_id"], real_row["pdf_url"])
print(result_normal)
assert result_normal.status == DocumentStatus.SUCCESS.value
assert result_normal.text_char_count > 1000
assert result_normal.is_scanned is False
print(f"Test 1 passed: real text-based PDF extracted via {result_normal.extraction_method} "
      f"({result_normal.text_char_count} chars, {result_normal.page_count} pages).")


## 14. Test 2 — Empty PDF (blank page: no text, no images)

In [ ]:
result_empty = process_document("test-empty-pdf", None, raw_bytes_override=_empty_bytes)
print(result_empty)
assert result_empty.status == DocumentStatus.EMPTY.value
assert result_empty.page_count == 1
assert result_empty.text_char_count == 0
print("Test 2 passed: blank PDF classified as \'empty\' (not miscategorized as \'scanned\'), not a crash.")


## 15. Test 3 — Corrupted PDF

In [ ]:
result_corrupted = process_document("test-corrupted-pdf", None, raw_bytes_override=_corrupted_bytes)
print(result_corrupted)
assert result_corrupted.status == DocumentStatus.CORRUPTED.value
assert result_corrupted.local_path is None, "A corrupted file should not be saved as if it were valid."
print("Test 3 passed: corrupted PDF classified as \'corrupted\', not saved, not a crash.")


## 16. Test 4 — Password-Protected PDF

In [ ]:
result_password = process_document("test-password-pdf", None, raw_bytes_override=_password_bytes)
print(result_password)
assert result_password.status == DocumentStatus.PASSWORD_PROTECTED.value
assert result_password.text_char_count == 0
print("Test 4 passed: encrypted PDF detected and classified as \'password_protected\' before any extraction was attempted.")


## 17. Test 5 — Missing PDF (404) & Network Failure

Two real, non-mocked network scenarios: a URL on DGCA's own domain that
doesn't exist, and a connection attempt to a non-routable IP address
(simulating "DGCA website unavailable").


In [ ]:
result_missing = process_document(
    "test-missing-pdf",
    "https://www.dgca.gov.in/digigov-portal/this-path-does-not-exist-12345",
)
print(result_missing)
assert result_missing.status == DocumentStatus.DOWNLOAD_FAILED.value
print("Missing-PDF (404) test passed.")

result_network = process_document("test-network-failure", "http://10.255.255.1/unreachable.pdf")
print(result_network)
assert result_network.status == DocumentStatus.DOWNLOAD_FAILED.value
print("Network-failure test passed.")
print("Test 5 passed.")


## 18. Test 6 — Real Scanned PDF (OCR gating)

A genuine DGCA tender document that Phase 2/4 inspection already confirmed
extracts to zero characters via PyMuPDF despite having real pages -- a
naturally occurring scanned document, no synthetic fixture needed. Since
`OCR_AVAILABLE` is `False` in this environment, the expected outcome is
`needs_ocr`, not a crash and not a false "success".


In [ ]:
scanned_row = conn.execute(
    "SELECT publication_id, pdf_url FROM publications WHERE publication_id = %s",
    ("tender-eVzkArZYTgSmhfx9huNw",),
).fetchone()

result_scanned = process_document(scanned_row["publication_id"], scanned_row["pdf_url"])
print(result_scanned)
assert result_scanned.is_scanned is True
if OCR_AVAILABLE:
    assert result_scanned.status == DocumentStatus.OCR_SUCCESS.value
    print(f"Test 6 passed: scanned PDF OCR\'d successfully ({result_scanned.text_char_count} chars).")
else:
    assert result_scanned.status == DocumentStatus.NEEDS_OCR.value
    assert result_scanned.text_char_count == 0
    print("Test 6 passed: scanned PDF correctly detected and classified \'needs_ocr\' (OCR unavailable here, handled gracefully).")


## 19. Batch Run — all Phase 3 records, `document_hash` written back

Processes every real publication currently in the Phase 3 database and
writes each one's `document_hash` back, closing the loop Phase 3 left open.


In [ ]:
all_rows = conn.execute("SELECT publication_id, pdf_url FROM publications").fetchall()
batch_results: List[DocumentProcessingResult] = []

for row in all_rows:
    res = process_document(row["publication_id"], row["pdf_url"])
    batch_results.append(res)
    if res.document_hash:
        update_document_hash(conn, row["publication_id"], res.document_hash)

status_counts: Dict[str, int] = {}
for res in batch_results:
    status_counts[res.status] = status_counts.get(res.status, 0) + 1

print(f"Processed {len(batch_results)} publications.")
print(f"Status breakdown: {status_counts}")


## 20. Results

In [ ]:
print("=" * 70)
print("DOCUMENT PROCESSING RESULTS")
print("=" * 70)
for res in batch_results:
    size_kb = f"{res.file_size_bytes / 1024:.0f} KB" if res.file_size_bytes else "n/a"
    print(f"- [{res.status:<18}] {res.publication_id:<40} {size_kb:>10}  chars={res.text_char_count}")
print("=" * 70)
print(f"documents saved to : {DOCUMENTS_DIR}")
print(f"database updated   : PostgreSQL (DATABASE_URL)")

conn.close()


## 21. Next Steps

**Phase 4 status: PDF download, validation, text extraction, and OCR gating
built and tested** against real DGCA PDFs (both text-based and genuinely
scanned) plus synthetic edge cases (empty, corrupted, password-protected)
and two real network failure modes (404, unreachable host).

### Known limitation

- OCR is implemented but inactive in this environment (Tesseract not
  installed) -- scanned documents are correctly flagged `needs_ocr` rather
  than silently skipped. Installing Tesseract (see Section 2) activates it
  with no code changes.

### Not started yet (by design)

- Phase 5: Old-vs-new AI comparison (will consume the extracted text this
  notebook saves to `data/documents/*.txt`)
- Phase 6: AI summary integration into the scraped-data flow
- Phase 7: Agentic workflow orchestration
- Phase 8: Streamlit dashboard

Waiting for confirmation before starting Phase 5.
